In [3]:
import pandas as pd
m = pd.read_parquet("/Users/benjaminsalop/Desktop/Oxford/Research/edca/inputs/canonical/materials.parquet")
print("materials rows:", len(m))
print("columns:", m.columns.tolist())
# list candidate carbon columns we expect (adjust names if your code uses other names)
expected = [
    "ec_a1a3_per_m3", "ec_a1a3_per_kg", "ec_a4_per_kg", "density_kg_m3",
    "kg_per_m3", "carbon_per_kg", "carbon_per_m3"
]
for col in expected:
    if col in m.columns:
        s = m[col]
        print(f"{col}: present, non-null count {s.notna().sum()}, null count {s.isna().sum()}")
    else:
        print(f"{col}: MISSING")
# show first few rows
print(m.head(10).to_string())


materials rows: 22
columns: ['material_id', 'family', 'standard_grade', 'concrete_f_ck', 'steel_fy', 'steel_fu', 'density', 'unit', 'ec_a1a3_volumetric', 'ec_a1a3_mass', 'ec_a4_ton_km', 'transport_distance', 'ec_a5_mass', 'cost_volume', 'cost_mass']
ec_a1a3_per_m3: MISSING
ec_a1a3_per_kg: MISSING
ec_a4_per_kg: MISSING
density_kg_m3: MISSING
kg_per_m3: MISSING
carbon_per_kg: MISSING
carbon_per_m3: MISSING
      material_id    family  standard_grade  concrete_f_ck  steel_fy  steel_fu   density      unit  ec_a1a3_volumetric  ec_a1a3_mass  ec_a4_ton_km  transport_distance  ec_a5_mass  cost_volume  cost_mass
0        concrete  concrete             NaN           40.0       NaN       NaN    24.000    metric             330.000         0.138          0.06                  50       0.005   120.000000       0.05
1          screed  concrete             NaN           40.0       NaN       NaN    20.000    metric             330.000         0.138          0.06                  50       0.005   120.0

In [6]:
import pandas as pd
from edca_code.scripts.core import carbon
m = carbon.load_materials_table("/Users/benjaminsalop/Desktop/Oxford/Research/edca/inputs/source/presets/materials/materials.csv")
print("rows:", len(m))
print(m[["density_kg_per_m3","ec_a1a3_volumetric","ec_a1a3_mass","ec_a4_per_ton_km","transport_km"]].head(10).to_string())


rows: 22
                density_kg_per_m3  ec_a1a3_volumetric  ec_a1a3_mass  ec_a4_per_ton_km  transport_km
material_id                                                                                        
concrete                 2400.000             330.000         0.138              0.06            50
screed                   2000.000             330.000         0.138              0.06            50
pt                       7850.000                 NaN         2.000              0.06           200
steel                    7850.000                 NaN         2.000              0.06           200
rebar                    7850.000                 NaN         2.000              0.06           200
us_nw_concrete           2322.677             319.368         0.138              0.06            50
us_lw_concrete           1762.000             242.275         0.138              0.06            50
us_steel_22              7850.000                 NaN         2.000              0.06      

In [7]:
from edca_code.scripts.core.takeoff import bom_per_m2_from_system_row
from edca_code.scripts.core.carbon import compute_assembly_carbon_from_bom, load_materials_table
import pandas as pd
systems = pd.read_parquet("/Users/benjaminsalop/Desktop/Oxford/Research/edca/inputs/canonical/system_variants.parquet")
materials = load_materials_table("/Users/benjaminsalop/Desktop/Oxford/Research/edca/inputs/source/presets/materials/materials.csv")
row = systems[systems["system_variant"]=="TR60_normal_weight_10"].iloc[0]
bom = bom_per_m2_from_system_row(row)
res = compute_assembly_carbon_from_bom(bom, materials)
print("per_material example (first 10):", res["per_material"][:10])
print("totals:", res["totals"])


per_material example (first 10): [{'material_id': 'concrete', 'category': 'concrete', 'qty_m3': 0.13541667, 'qty_kg': 325.000008, 'a1a3': 44.6875011, 'a4': 0.9750000239999999, 'a5': 1.62500004, 'total': 47.287501164000005, 'cost': 16.250000399999998}, {'material_id': 'eu_steel_1.2', 'category': 'steel_m3', 'qty_m3': 0.065, 'qty_kg': 510.25, 'a1a3': 1020.5, 'a4': 6.122999999999999, 'a5': 10.205, 'total': 1036.828, 'cost': 1.6560505e-05}, {'material_id': 'timber', 'category': 'timber', 'qty_m3': nan, 'qty_kg': 0.0, 'a1a3': 0.0, 'a4': 0.0, 'a5': 0.0, 'total': 0.0, 'cost': 0.0}, {'material_id': 'screed', 'category': 'concrete', 'qty_m3': nan, 'qty_kg': nan, 'a1a3': nan, 'a4': 0.0, 'a5': nan, 'total': nan, 'cost': nan}]
totals: {'total_a1a3': nan, 'total_a4': 7.098000023999999, 'total_a5': nan, 'total': nan, 'total_cost': nan}
